# Reproduction of Adult dataset experiments

In this notebook we reproduce the results from Table 2 of the DECAF paper. We compare various methods for generating debiased data using the DECAF model against synthetic data generated using benchmark models GAN, WGAN-GP and FairGAN. As described in the paper we run all experiments (as implemented in this notebook) 10 times and avarage the results.

In [1]:
with open('repNum.txt',"r") as f:
    repNum = f.read().strip()
    repNum = float(repNum)

In [2]:
repNum

9.0

In [3]:
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.metrics import confusion_matrix


from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

from data import load_adult, preprocess_adult
from metrics import DP, FTU
from train import train_decaf, train_fairgan, train_vanilla_gan, train_wgan_gp


## Loading data

In [4]:
dataset = load_adult()
dataset.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516.0,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=50K
1,50,Self-emp-not-inc,83311.0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,13.0,United-States,<=50K
2,38,Private,215646.0,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=50K
3,53,Private,234721.0,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.0,0.0,40.0,United-States,<=50K
4,28,Private,338409.0,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,<=50K


Preprocess the data next in order to make it suitable for training models on.

In [5]:
dataset = preprocess_adult(dataset)
dataset.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,0.301370,0.833333,0.043350,0.000000,0.800000,0.333333,0.615385,0.6,0.0,1.0,0.02174,0.0,0.397959,0.0,1.0
1,0.452055,0.166667,0.047274,0.000000,0.800000,0.000000,0.307692,0.4,0.0,1.0,0.00000,0.0,0.122449,0.0,1.0
2,0.287671,0.000000,0.136877,0.200000,0.533333,0.166667,0.461538,0.6,0.0,1.0,0.00000,0.0,0.397959,0.0,1.0
3,0.493151,0.000000,0.149792,0.133333,0.400000,0.000000,0.461538,0.4,1.0,1.0,0.00000,0.0,0.397959,0.0,1.0
4,0.150685,0.000000,0.219998,0.000000,0.800000,0.000000,0.384615,0.0,1.0,0.0,0.00000,0.0,0.397959,0.3,1.0


Split the dataset into train and test folds. Test fold size is 2000.

In [6]:
# Split data into train and testing sets
dataset_train, dataset_test = train_test_split(dataset, test_size=2000,
                                               stratify=dataset['income'])

print('Size of train set:', len(dataset_train))
print('Size of test set:', len(dataset_test))

Size of train set: 43222
Size of test set: 2000


### Defining the DAG

We need to define a DAG which captures the biases of the dataset. As described in the DECAF paper normally a causal discovery algorithm is used. In this notebook we simply copy the DAG which as described in the Zhang et al. paper which is the one also used in the DECAF paper.

In [ ]:
# Define DAG for Adult dataset
dag = [
    # Edges from race
    ['race', 'occupation'],
    ['race', 'income'],
    ['race', 'hours-per-week'],
    ['race', 'education'],
    ['race', 'marital-status'],

    # Edges from age
    ['age', 'occupation'],
    ['age', 'hours-per-week'],
    ['age', 'income'],
    ['age', 'workclass'],
    ['age', 'marital-status'],
    ['age', 'education'],
    ['age', 'relationship'],
    
    # Edges from sex
    ['sex', 'occupation'],
    ['sex', 'marital-status'],
    ['sex', 'income'],
    ['sex', 'workclass'],
    ['sex', 'education'],
    ['sex', 'relationship'],
    
    # Edges from native country
    ['native-country', 'marital-status'],
    ['native-country', 'hours-per-week'],
    ['native-country', 'education'],
    ['native-country', 'workclass'],
    ['native-country', 'income'],
    ['native-country', 'relationship'],
    
    # Edges from marital status
    ['marital-status', 'occupation'],
    ['marital-status', 'hours-per-week'],
    ['marital-status', 'income'],
    ['marital-status', 'workclass'],
    ['marital-status', 'relationship'],
    ['marital-status', 'education'],
    
    # Edges from education
    ['education', 'occupation'],
    ['education', 'hours-per-week'],
    ['education', 'income'],
    ['education', 'workclass'],
    ['education', 'relationship'],
    
    # All remaining edges
    ['occupation', 'income'],
    ['hours-per-week', 'income'],
    ['workclass', 'income'],
    ['relationship', 'income'],
]

def dag_to_idx(df, dag):
    """Convert columns in a DAG to the corresponding indices."""

    dag_idx = []
    for edge in dag:
        dag_idx.append([df.columns.get_loc(edge[0]), df.columns.get_loc(edge[1])])

    return dag_idx

# Convert the DAG to one that can be provided to the DECAF model
dag_seed = dag_to_idx(dataset, dag)
print(dag_seed)

It's also necessary to define edges we want to remove from the DAG in order to meet the various fairness criteria described in the paper.

In [ ]:
def create_bias_dict(df, edge_map):
    """
    Convert the given edge tuples to a bias dict used for generating
    debiased synthetic data.
    """
    bias_dict = {}
    for key, val in edge_map.items():
        bias_dict[df.columns.get_loc(key)] = [df.columns.get_loc(f) for f in val]
    
    return bias_dict

# Bias dictionary to satisfy FTU
bias_dict_ftu = create_bias_dict(dataset, {'income': ['sex']})
print('Bias dict FTU:', bias_dict_ftu)

# Bias dictionary to satisfy DP
bias_dict_dp = create_bias_dict(dataset, {'income': [
    'occupation', 'hours-per-week', 'marital-status', 'education', 'sex',
    'workclass', 'relationship']})
print('Bias dict DP:', bias_dict_dp)

# Bias dictionary to satisfy CF
bias_dict_cf = create_bias_dict(dataset, {'income': [
    'marital-status', 'sex']})
print('Bias dict CF:', bias_dict_cf)

## Experiments

We have loaded and preprocessed the data and we are ready to run the experiments. For each experiment we train a generative model, sample synthetic data from the trained model and then obtain metrics by training and evaluating a downstream multi-layer perceptron using the test fold we generated in the previous section. We use the MLP model from `sklearn` with default parameters which matches the settings described in Appendix D of the paper.

In [21]:
def eval_model(dataset_train, dataset_test):
    """Helper function that prints evaluation metrics."""

    X_train, y_train = dataset_train.drop(columns=['income']), dataset_train['income']
    X_test, y_test = dataset_test.drop(columns=['income']), dataset_test['income']

    clf = MLPClassifier(max_iter=1000)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    auroc = roc_auc_score(y_test, y_pred)
    dp = DP(clf, X_test)
    ftu = FTU(clf, X_test)
    conf = confusion_matrix(y_test, y_pred)

    return {'precision': precision, 'recall': recall, 'auroc': auroc,
            'dp': dp, 'ftu': ftu, "confMatrix":conf}

Make the result dict:

In [ ]:
results = {}

### Original dataset

As a benchmark we want to first train the downstream model on the original dataset.

In [ ]:
res = eval_model(dataset_train, dataset_test)
print(res)

In [ ]:
results['original'] = res

In the following sections we train various models in order to reproduce the results from Table 2 of the DECAF paper.

### GAN

In [ ]:
synth_data = train_vanilla_gan(dataset_train)
synth_data.head()

In [ ]:
res = eval_model(synth_data, dataset_test)
print(res)

In [ ]:
results['GAN'] = res

### WGAN-GP

In [ ]:
synth_data = train_wgan_gp(dataset_train)
synth_data.head()

In [ ]:
res = eval_model(synth_data, dataset_test)
print(res)

In [ ]:
results['WGAN-GP'] = res

### FairGAN

In [ ]:
synth_data = train_fairgan(dataset_train)
synth_data.head()

In [ ]:
res = eval_model(synth_data, dataset_test)
print(res)

In [ ]:
results['FairGAN'] = res

### DECAF

#### DECAF-ND

In [ ]:
synth_data = train_decaf(dataset_train, dag_seed)
synth_data.head()

In [ ]:
res = eval_model(synth_data, dataset_test)
print(res)

In [ ]:
results['DECAF-ND'] = res

#### DECAF-FTU

In [ ]:
synth_data = train_decaf(dataset_train, dag_seed, biased_edges=bias_dict_ftu)
synth_data.head()

In [ ]:
res = eval_model(synth_data, dataset_test)
print(res)

In [ ]:
results['DECAF-FTU'] = res

#### DECAF-CF

In [ ]:
synth_data = train_decaf(dataset_train, dag_seed, biased_edges=bias_dict_cf)
synth_data.head()

In [ ]:
res = eval_model(synth_data, dataset_test)
print(res)

In [ ]:
results['DECAF-CF'] = res

#### DECAF-DP

In [ ]:
synth_data = train_decaf(dataset_train, dag_seed, biased_edges=bias_dict_dp)
synth_data.head()

In [ ]:
res = eval_model(synth_data, dataset_test)

In [ ]:
results['DECAF-DP'] = res

#### SpGan

In [ ]:
from spgan import spgan

In [ ]:
from importlib import reload

In [ ]:
reload(spgan)

In [7]:
class optC:
    pass

In [8]:
opt = optC()
opt.latent_dim = 40
opt.batch_size = 64
opt.lr = 0.0005
opt.n_epochs = 500
opt.n_critic = 20
opt.clip_value = 0.01

In [ ]:
import pandas as pd
def helper(lam):
    opt.lam = lam
    res = spgan.train_spgan(dataset_train.to_numpy(),opt)
    q1 = pd.DataFrame(res)
    q1.columns = dataset_train.columns
    q1['income'] = q1['income']>0.5
    res = eval_model(q1, dataset_test)
    print(res)
    return res

### Run at multiple lambdas

In [ ]:
results["spgan-0.0"] = helper(0.0)
results["spgan-0.001"] = helper(0.001)
results["spgan-0.01"] = helper(0.01)
results["spgan-0.025"] = helper(0.025)
results["spgan-0.05"] = helper(0.05)
results["spgan-0.075"] = helper(0.075)
results["spgan-0.1"] = helper(0.1)
results["spgan-1"] = helper(1.0)

In [14]:
from ifgan import ifgan
from importlib import reload
reload(ifgan)

<module 'ifgan.ifgan' from '/Users/andrewelliott/Documents/hsbcWork/mahedPaper/FairSyn/UvA_FACT2022-main/ifgan/ifgan.py'>

In [17]:
opt.lam = 0.1
opt.clip_value = 0.05
q1 = ifgan.train_ifgan(dataset_train.to_numpy(),opt)

[Epoch 0/500] [Batch 0/676] [D loss: -0.049243] [G loss: 0.026962] [d1 loss: 0.747151] [d2 loss: 0.704183]  [sq d1 d2: 0.001846]  [num0: 22.000000] [num1 42.000000]
[Epoch 0/500] [Batch 20/676] [D loss: -0.193285] [G loss: 0.183723] [d1 loss: 0.700489] [d2 loss: 0.664577]  [sq d1 d2: 0.001290]  [num0: 20.000000] [num1 44.000000]
[Epoch 0/500] [Batch 40/676] [D loss: -0.396805] [G loss: 0.341277] [d1 loss: 0.643700] [d2 loss: 0.608600]  [sq d1 d2: 0.001232]  [num0: 15.000000] [num1 49.000000]
[Epoch 0/500] [Batch 60/676] [D loss: -0.684105] [G loss: 0.530043] [d1 loss: 0.596816] [d2 loss: 0.577702]  [sq d1 d2: 0.000365]  [num0: 16.000000] [num1 48.000000]
[Epoch 0/500] [Batch 80/676] [D loss: -1.064522] [G loss: 0.738464] [d1 loss: 0.542895] [d2 loss: 0.531623]  [sq d1 d2: 0.000127]  [num0: 14.000000] [num1 50.000000]
[Epoch 0/500] [Batch 100/676] [D loss: -1.227604] [G loss: 0.944726] [d1 loss: 0.507724] [d2 loss: 0.503691]  [sq d1 d2: 0.000016]  [num0: 13.000000] [num1 51.000000]
[Epo

[Epoch 1/500] [Batch 340/676] [D loss: -0.636954] [G loss: 1.250431] [d1 loss: 0.001568] [d2 loss: 0.001720]  [sq d1 d2: 0.000000]  [num0: 0.000000] [num1 64.000000]
[Epoch 1/500] [Batch 360/676] [D loss: -0.476983] [G loss: 1.241270] [d1 loss: 0.001565] [d2 loss: 0.001755]  [sq d1 d2: 0.000000]  [num0: 0.000000] [num1 64.000000]
[Epoch 1/500] [Batch 380/676] [D loss: -0.445795] [G loss: 1.220881] [d1 loss: 0.001451] [d2 loss: 0.001550]  [sq d1 d2: 0.000000]  [num0: 0.000000] [num1 64.000000]
[Epoch 1/500] [Batch 400/676] [D loss: -0.444247] [G loss: 1.211636] [d1 loss: 0.001437] [d2 loss: 0.001554]  [sq d1 d2: 0.000000]  [num0: 0.000000] [num1 64.000000]
[Epoch 1/500] [Batch 420/676] [D loss: -0.478622] [G loss: 1.175015] [d1 loss: 0.001450] [d2 loss: 0.001540]  [sq d1 d2: 0.000000]  [num0: 0.000000] [num1 64.000000]
[Epoch 1/500] [Batch 440/676] [D loss: -0.422531] [G loss: 1.165490] [d1 loss: 0.001430] [d2 loss: 0.001510]  [sq d1 d2: 0.000000]  [num0: 0.000000] [num1 64.000000]
[Epo

[Epoch 3/500] [Batch 20/676] [D loss: -0.143947] [G loss: 0.164688] [d1 loss: 0.103544] [d2 loss: 0.105862]  [sq d1 d2: 0.000005]  [num0: 1.000000] [num1 63.000000]
[Epoch 3/500] [Batch 40/676] [D loss: -0.138425] [G loss: 0.151547] [d1 loss: 0.003228] [d2 loss: 0.004146]  [sq d1 d2: 0.000001]  [num0: 0.000000] [num1 64.000000]
[Epoch 3/500] [Batch 60/676] [D loss: -0.183289] [G loss: 0.171339] [d1 loss: 0.176501] [d2 loss: 0.176190]  [sq d1 d2: 0.000000]  [num0: 2.000000] [num1 62.000000]
[Epoch 3/500] [Batch 80/676] [D loss: -0.132885] [G loss: 0.181400] [d1 loss: 0.009071] [d2 loss: 0.011227]  [sq d1 d2: 0.000005]  [num0: 0.000000] [num1 64.000000]
[Epoch 3/500] [Batch 100/676] [D loss: -0.175970] [G loss: 0.219149] [d1 loss: 0.011522] [d2 loss: 0.013540]  [sq d1 d2: 0.000004]  [num0: 0.000000] [num1 64.000000]
[Epoch 3/500] [Batch 120/676] [D loss: -0.134821] [G loss: 0.242897] [d1 loss: 0.156925] [d2 loss: 0.155993]  [sq d1 d2: 0.000001]  [num0: 2.000000] [num1 62.000000]
[Epoch 3

[Epoch 4/500] [Batch 380/676] [D loss: -0.101742] [G loss: 0.214328] [d1 loss: 0.249015] [d2 loss: 0.294918]  [sq d1 d2: 0.002107]  [num0: 6.000000] [num1 58.000000]
[Epoch 4/500] [Batch 400/676] [D loss: -0.110314] [G loss: 0.196634] [d1 loss: 0.263813] [d2 loss: 0.315481]  [sq d1 d2: 0.002670]  [num0: 7.000000] [num1 57.000000]
[Epoch 4/500] [Batch 420/676] [D loss: -0.084561] [G loss: 0.185766] [d1 loss: 0.201610] [d2 loss: 0.225696]  [sq d1 d2: 0.000580]  [num0: 4.000000] [num1 60.000000]
[Epoch 4/500] [Batch 440/676] [D loss: -0.106705] [G loss: 0.187556] [d1 loss: 0.234751] [d2 loss: 0.286618]  [sq d1 d2: 0.002690]  [num0: 6.000000] [num1 58.000000]
[Epoch 4/500] [Batch 460/676] [D loss: -0.111224] [G loss: 0.214834] [d1 loss: 0.228673] [d2 loss: 0.278829]  [sq d1 d2: 0.002516]  [num0: 6.000000] [num1 58.000000]
[Epoch 4/500] [Batch 480/676] [D loss: -0.165818] [G loss: 0.192678] [d1 loss: 0.202169] [d2 loss: 0.264260]  [sq d1 d2: 0.003855]  [num0: 6.000000] [num1 58.000000]
[Epo

[Epoch 6/500] [Batch 60/676] [D loss: -0.091254] [G loss: 0.243146] [d1 loss: 0.028931] [d2 loss: 0.029386]  [sq d1 d2: 0.000000]  [num0: 0.000000] [num1 64.000000]
[Epoch 6/500] [Batch 80/676] [D loss: -0.060869] [G loss: 0.222619] [d1 loss: 0.053783] [d2 loss: 0.064552]  [sq d1 d2: 0.000116]  [num0: 1.000000] [num1 63.000000]
[Epoch 6/500] [Batch 100/676] [D loss: -0.115902] [G loss: 0.259873] [d1 loss: 0.104900] [d2 loss: 0.095408]  [sq d1 d2: 0.000090]  [num0: 2.000000] [num1 62.000000]
[Epoch 6/500] [Batch 120/676] [D loss: -0.110989] [G loss: 0.279974] [d1 loss: 0.065119] [d2 loss: 0.073562]  [sq d1 d2: 0.000071]  [num0: 1.000000] [num1 63.000000]
[Epoch 6/500] [Batch 140/676] [D loss: -0.118016] [G loss: 0.295837] [d1 loss: 0.015050] [d2 loss: 0.014433]  [sq d1 d2: 0.000000]  [num0: 0.000000] [num1 64.000000]
[Epoch 6/500] [Batch 160/676] [D loss: -0.102624] [G loss: 0.322686] [d1 loss: 0.145113] [d2 loss: 0.133079]  [sq d1 d2: 0.000145]  [num0: 2.000000] [num1 62.000000]
[Epoch

...

[Epoch 493/500] [Batch 100/676] [D loss: 0.007655] [G loss: -0.021762] [d1 loss: 0.010397] [d2 loss: 0.009728]  [sq d1 d2: 0.000000]  [num0: 15.000000] [num1 49.000000]
[Epoch 493/500] [Batch 120/676] [D loss: 0.003667] [G loss: -0.001974] [d1 loss: 0.077194] [d2 loss: 0.086824]  [sq d1 d2: 0.000093]  [num0: 17.000000] [num1 47.000000]
[Epoch 493/500] [Batch 140/676] [D loss: -0.000976] [G loss: 0.015747] [d1 loss: 0.030205] [d2 loss: 0.052056]  [sq d1 d2: 0.000477]  [num0: 13.000000] [num1 51.000000]
[Epoch 493/500] [Batch 160/676] [D loss: -0.004139] [G loss: 0.026015] [d1 loss: 0.044594] [d2 loss: 0.042327]  [sq d1 d2: 0.000005]  [num0: 18.000000] [num1 46.000000]
[Epoch 493/500] [Batch 180/676] [D loss: -0.010403] [G loss: 0.036163] [d1 loss: 0.026289] [d2 loss: 0.031209]  [sq d1 d2: 0.000024]  [num0: 20.000000] [num1 44.000000]
[Epoch 493/500] [Batch 200/676] [D loss: -0.025780] [G loss: 0.052738] [d1 loss: 0.066359] [d2 loss: 0.078764]  [sq d1 d2: 0.000154]  [num0: 15.000000] [nu

[Epoch 494/500] [Batch 460/676] [D loss: -0.026310] [G loss: -0.003298] [d1 loss: 0.017902] [d2 loss: 0.022890]  [sq d1 d2: 0.000025]  [num0: 19.000000] [num1 45.000000]
[Epoch 494/500] [Batch 480/676] [D loss: -0.037798] [G loss: 0.044668] [d1 loss: 0.050330] [d2 loss: 0.039751]  [sq d1 d2: 0.000112]  [num0: 18.000000] [num1 46.000000]
[Epoch 494/500] [Batch 500/676] [D loss: 0.017218] [G loss: 0.009374] [d1 loss: 0.064388] [d2 loss: 0.077799]  [sq d1 d2: 0.000180]  [num0: 14.000000] [num1 50.000000]
[Epoch 494/500] [Batch 520/676] [D loss: -0.002290] [G loss: -0.014214] [d1 loss: 0.026853] [d2 loss: 0.024821]  [sq d1 d2: 0.000004]  [num0: 14.000000] [num1 50.000000]
[Epoch 494/500] [Batch 540/676] [D loss: 0.013173] [G loss: 0.014052] [d1 loss: 0.039462] [d2 loss: 0.051860]  [sq d1 d2: 0.000154]  [num0: 16.000000] [num1 48.000000]
[Epoch 494/500] [Batch 560/676] [D loss: 0.002150] [G loss: 0.023873] [d1 loss: 0.096221] [d2 loss: 0.119353]  [sq d1 d2: 0.000535]  [num0: 17.000000] [num

[Epoch 496/500] [Batch 140/676] [D loss: 0.002875] [G loss: -0.062951] [d1 loss: 0.083070] [d2 loss: 0.092054]  [sq d1 d2: 0.000081]  [num0: 14.000000] [num1 50.000000]
[Epoch 496/500] [Batch 160/676] [D loss: 0.000140] [G loss: -0.070653] [d1 loss: 0.060513] [d2 loss: 0.054526]  [sq d1 d2: 0.000036]  [num0: 18.000000] [num1 46.000000]
[Epoch 496/500] [Batch 180/676] [D loss: -0.002559] [G loss: -0.063129] [d1 loss: 0.118099] [d2 loss: 0.140021]  [sq d1 d2: 0.000481]  [num0: 15.000000] [num1 49.000000]
[Epoch 496/500] [Batch 200/676] [D loss: -0.008208] [G loss: -0.045638] [d1 loss: 0.045749] [d2 loss: 0.055729]  [sq d1 d2: 0.000100]  [num0: 16.000000] [num1 48.000000]
[Epoch 496/500] [Batch 220/676] [D loss: -0.006111] [G loss: -0.040218] [d1 loss: 0.085630] [d2 loss: 0.094868]  [sq d1 d2: 0.000085]  [num0: 19.000000] [num1 45.000000]
[Epoch 496/500] [Batch 240/676] [D loss: -0.028663] [G loss: -0.029925] [d1 loss: 0.128769] [d2 loss: 0.134385]  [sq d1 d2: 0.000032]  [num0: 17.000000]

[Epoch 497/500] [Batch 500/676] [D loss: 0.003439] [G loss: -0.120432] [d1 loss: 0.040383] [d2 loss: 0.041814]  [sq d1 d2: 0.000002]  [num0: 22.000000] [num1 42.000000]
[Epoch 497/500] [Batch 520/676] [D loss: 0.021710] [G loss: -0.097440] [d1 loss: 0.126607] [d2 loss: 0.129605]  [sq d1 d2: 0.000009]  [num0: 16.000000] [num1 48.000000]
[Epoch 497/500] [Batch 540/676] [D loss: 0.010302] [G loss: -0.080546] [d1 loss: 0.035112] [d2 loss: 0.025320]  [sq d1 d2: 0.000096]  [num0: 16.000000] [num1 48.000000]
[Epoch 497/500] [Batch 560/676] [D loss: 0.000382] [G loss: -0.079646] [d1 loss: 0.080489] [d2 loss: 0.093124]  [sq d1 d2: 0.000160]  [num0: 14.000000] [num1 50.000000]
[Epoch 497/500] [Batch 580/676] [D loss: -0.008638] [G loss: -0.080509] [d1 loss: 0.047180] [d2 loss: 0.029113]  [sq d1 d2: 0.000326]  [num0: 13.000000] [num1 51.000000]
[Epoch 497/500] [Batch 600/676] [D loss: -0.020620] [G loss: -0.064760] [d1 loss: 0.064641] [d2 loss: 0.054479]  [sq d1 d2: 0.000103]  [num0: 17.000000] [

[Epoch 499/500] [Batch 160/676] [D loss: -0.020882] [G loss: 0.150318] [d1 loss: 0.057749] [d2 loss: 0.056483]  [sq d1 d2: 0.000002]  [num0: 18.000000] [num1 46.000000]
[Epoch 499/500] [Batch 180/676] [D loss: -0.049311] [G loss: 0.137348] [d1 loss: 0.010778] [d2 loss: 0.007305]  [sq d1 d2: 0.000012]  [num0: 16.000000] [num1 48.000000]
[Epoch 499/500] [Batch 200/676] [D loss: 0.013969] [G loss: 0.101921] [d1 loss: 0.051689] [d2 loss: 0.052032]  [sq d1 d2: 0.000000]  [num0: 17.000000] [num1 47.000000]
[Epoch 499/500] [Batch 220/676] [D loss: -0.000090] [G loss: 0.136794] [d1 loss: 0.034458] [d2 loss: 0.043019]  [sq d1 d2: 0.000073]  [num0: 19.000000] [num1 45.000000]
[Epoch 499/500] [Batch 240/676] [D loss: 0.036218] [G loss: 0.091719] [d1 loss: 0.102958] [d2 loss: 0.096559]  [sq d1 d2: 0.000041]  [num0: 16.000000] [num1 48.000000]
[Epoch 499/500] [Batch 260/676] [D loss: 0.011624] [G loss: 0.079865] [d1 loss: 0.025998] [d2 loss: 0.030245]  [sq d1 d2: 0.000018]  [num0: 16.000000] [num1 

In [22]:
import pandas as pd
q1 = pd.DataFrame(q1)
q1.columns = dataset_train.columns
q1['income'] = q1['income']>0.5
res = eval_model(q1, dataset_test)
print(res)

{'precision': 0.812200956937799, 'recall': 0.9029255319148937, 'auroc': 0.6349305078929307, 'dp': 0.11800063453606757, 'ftu': 0.014000000000000012, 'confMatrix': array([[ 182,  314],
       [ 146, 1358]])}


## Save data

In [ ]:
import pickle

In [ ]:
with open("results/data_"+str(repNum)+'.p','wb') as f:
    pickle.dump(results,f)